# Классификатор «очки / без очков» (этап 2 двухэтапного детектора)
PyTorch, 3 свёрточных блока + BatchNorm + Pooling, Dropout, аугментация, экспорт в ONNX.
Вход сети: 1x3x96x96, RGB, значения 0..1 (без mean/std). Выход: вероятности (Softmax внутри графа).

In [ ]:
!pip -q install onnx onnxscript onnxruntime
import os, zipfile
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from google.colab import files

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Загрузка датасета (dataset.zip, собранного collect_data.py)

In [ ]:
up = files.upload()   # выбери dataset.zip
with zipfile.ZipFile("dataset.zip") as z:
    z.extractall(".")
for c in sorted(os.listdir("dataset")):
    print(c, len(os.listdir(f"dataset/{c}")))

## 2. Аугментация и разбиение 80/20

In [ ]:
IMG = 96
train_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomRotation(15),                                        # поворот ±15°
    transforms.RandomAffine(0, translate=(0.1, 0.1), scale=(0.9, 1.1)),   # сдвиг до 10%, масштаб
    transforms.RandomHorizontalFlip(),                                    # отражение
    transforms.ColorJitter(brightness=0.3, contrast=0.3),                 # яркость/контраст
    transforms.ToTensor(),                                                # RGB, 0..1, CHW
])
val_tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor()])

full_train = datasets.ImageFolder("dataset", train_tf)
full_val   = datasets.ImageFolder("dataset", val_tf)
classes = full_train.classes
print("classes:", classes)   # порядок индексов -> должен совпасть с labels в C++

g = torch.Generator().manual_seed(42)
perm = torch.randperm(len(full_train), generator=g).tolist()
n_train = int(0.8 * len(perm))
train_ds = Subset(full_train, perm[:n_train])
val_ds   = Subset(full_val,   perm[n_train:])
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=64)
print("train:", len(train_ds), "val:", len(val_ds))

## 3. Архитектура: 3 блока Conv-BN-ReLU-MaxPool + классификатор с Dropout

In [ ]:
def block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1),
        nn.BatchNorm2d(cout),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )

model = nn.Sequential(
    block(3, 32), block(32, 64), block(64, 128),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(),
    nn.Dropout(0.4),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, len(classes)),
).to(device)
print(model)

## 4. Обучение

In [ ]:
EPOCHS = 25
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()
hist = {"loss": [], "val_loss": [], "acc": [], "val_acc": []}
best_acc, best_state = 0, None

def run(dl, train):
    model.train(train)
    tot_loss, correct, n = 0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            tot_loss += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            n += len(y)
    return tot_loss / n, correct / n

for ep in range(EPOCHS):
    l, a = run(train_dl, True)
    vl, va = run(val_dl, False)
    hist["loss"].append(l); hist["acc"].append(a)
    hist["val_loss"].append(vl); hist["val_acc"].append(va)
    if va > best_acc:
        best_acc = va
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    print(f"epoch {ep+1:2d}  loss {l:.3f} acc {a:.3f} | val_loss {vl:.3f} val_acc {va:.3f}")
print("best val acc:", best_acc)

## 5. Графики обучения

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist["loss"], label="train"); ax[0].plot(hist["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(hist["acc"], label="train"); ax[1].plot(hist["val_acc"], label="val")
ax[1].axhline(0.85, color="r", ls="--", label="0.85")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.show()

## 6. Экспорт в ONNX (Softmax внутри графа) и проверка

In [ ]:
model.load_state_dict(best_state)
model.eval()
export_model = nn.Sequential(model, nn.Softmax(dim=1)).cpu().eval()
dummy = torch.randn(1, 3, IMG, IMG)
torch.onnx.export(export_model, dummy, "glasses_cnn.onnx",
                  input_names=["input"], output_names=["probs"],
                  opset_version=13, dynamo=False)

# проверка через onnxruntime на валидации
import onnxruntime as ort
sess = ort.InferenceSession("glasses_cnn.onnx")
correct = n = 0
for x, y in val_dl:
    p = sess.run(None, {"input": x.numpy()})[0]
    correct += (p.argmax(1) == y.numpy()).sum(); n += len(y)
print("ONNX val accuracy:", correct / n, "| classes:", classes)
files.download("glasses_cnn.onnx")